# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruchitgoud/flyrankai-intern/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The ranked queue is intended to help a content team decide which pages should be reviewed first.

The main reason code is `STALE_AND_LOW_CTR`, which identifies content that is relatively old and has weak click-through performance. The corresponding action is `REVIEW_REFRESH`, meaning the page should be reviewed for a possible content refresh.

The ranking is a prioritization signal, not an automatic instruction. A high-ranked page should receive human review before any content change is made.

### Reason code → action

| Reason code | Action | Meaning |
|---|---|---|
| `STALE_AND_LOW_CTR` | `REVIEW_REFRESH` | Review older content with comparatively weak CTR |
| `LOW_CTR` | `REVIEW_CTR` | Review search-result presentation and relevance |
| `STALE` | `REVIEW_REFRESH` | Review older content for freshness and continued relevance |

The queue should be interpreted as "review this first" rather than "change this automatically."

In [1]:
import os
import pandas as pd

repo_path = "/content/flyrankai-intern"

if not os.path.exists(repo_path):
    !git clone https://github.com/ruchitgoud/flyrankai-intern.git

%cd /content/flyrankai-intern

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df["staleness_score"] = (
    df["days_since_last_update"]
    .rank(pct=True)
    .fillna(0)
)

ctr_median = df["ctr"].median()

df["low_ctr_score"] = (
    (ctr_median - df["ctr"])
    .clip(lower=0)
    .rank(pct=True)
    .fillna(0)
)

df["action_score"] = (
    0.60 * df["staleness_score"]
    + 0.40 * df["low_ctr_score"]
)

df["reason_code"] = "OTHER"
df["action"] = "REVIEW"

stale = df["days_since_last_update"] >= df["days_since_last_update"].median()
low_ctr = df["ctr"] < ctr_median

df.loc[stale & low_ctr, "reason_code"] = "STALE_AND_LOW_CTR"
df.loc[stale & low_ctr, "action"] = "REVIEW_REFRESH"

df.loc[~stale & low_ctr, "reason_code"] = "LOW_CTR"
df.loc[~stale & low_ctr, "action"] = "REVIEW_CTR"

df.loc[stale & ~low_ctr, "reason_code"] = "STALE"
df.loc[stale & ~low_ctr, "action"] = "REVIEW_REFRESH"

action_queue = (
    df[
        [
            "content_id",
            "action_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "ctr",
            "avg_position"
        ]
    ]
    .sort_values("action_score", ascending=False)
    .reset_index(drop=True)
)

action_queue["rank"] = action_queue.index + 1

print("\nRanked action queue created.")
print("Rows:", len(action_queue))

display(action_queue.head(10))

Cloning into 'flyrankai-intern'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 172 (delta 75), reused 95 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 1.88 MiB | 6.37 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/flyrankai-intern
Rows: 30000
Columns: 44

Ranked action queue created.
Rows: 30000


,content_id,action_score,reason_code,action,days_since_last_update,ctr,avg_position,rank
0,content_f6fdf87348f6,0.911907,STALE_AND_LOW_CTR,REVIEW_REFRESH,373,0.0,32.5,1
1,content_55a5b1c46474,0.911907,STALE_AND_LOW_CTR,REVIEW_REFRESH,373,0.0,7.5,2
2,content_1b4ec72dafd4,0.911857,STALE_AND_LOW_CTR,REVIEW_REFRESH,372,0.0,7.0,3
3,content_8d56efff1e71,0.911857,STALE_AND_LOW_CTR,REVIEW_REFRESH,372,0.0,35.0,4
4,content_06e19c6486b0,0.911797,STALE_AND_LOW_CTR,REVIEW_REFRESH,334,0.0,5.0,5
5,content_e2b702f4f92b,0.911797,STALE_AND_LOW_CTR,REVIEW_REFRESH,334,0.0,9.3,6
6,content_02b0d6e30129,0.911737,STALE_AND_LOW_CTR,REVIEW_REFRESH,313,0.0,6.9,7
7,content_7a888d3d99c8,0.911737,STALE_AND_LOW_CTR,REVIEW_REFRESH,313,0.0,67.6,8
8,content_94991fe6268c,0.911737,STALE_AND_LOW_CTR,REVIEW_REFRESH,313,0.0,12.4,9
9,content_6476d1d8c050,0.911737,STALE_AND_LOW_CTR,REVIEW_REFRESH,313,0.0,67.8,10


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

The action playbook is intended to help a content team prioritize pages for human review. The ranked score and reason code provide a consistent way to decide which pages may deserve attention first.

The main use is to identify pages that show signals such as staleness and comparatively low CTR, then guide a reviewer toward an appropriate content action.

### Limits

The score is a decision-support signal, not proof that a page needs a refresh or that a refresh will improve performance.

The recommendations are based on the available dataset, selected features, and the validation work completed in Weeks 5 and 6. They may not generalize to every client, content type, or future time period.

The playbook should not be used to automatically publish, rewrite, delete, or make major changes to content without human review.

In [2]:
print("Intended use: human-reviewed content prioritization")
print("Automatic content changes: NOT allowed")
print("Primary action:", "REVIEW_REFRESH")
print("Queue size:", len(action_queue))


Intended use: human-reviewed content prioritization
Automatic content changes: NOT allowed
Primary action: REVIEW_REFRESH
Queue size: 30000


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review

Every ranked recommendation should be reviewed by a person before any content action is taken.

The reviewer should check:

1. Whether the page is still relevant to the target audience and search intent.
2. Whether the low CTR or declining signal has a reasonable content-related explanation.
3. Whether the page has recently been changed or has other business or technical context that the model cannot see.
4. Whether the recommended action is appropriate for the page rather than simply following its rank.
5. Whether the proposed change can be tested and measured safely.

### No-go list

The model should never automatically:

- publish or rewrite content;
- delete or redirect a page;
- change important business, legal, or factual information;
- make decisions based only on the model score;
- treat a recommendation as proof that a refresh will improve performance.

The model's role is to prioritize pages for human review. Final content decisions remain with a qualified reviewer.

In [3]:
required_review_checks = [
    "Search intent and page relevance",
    "Reason for low CTR or declining signal",
    "Recent page changes and context",
    "Suitability of the recommended action",
    "Safe measurement after the action"
]

no_go_actions = [
    "Automatic publishing or rewriting",
    "Automatic deletion or redirection",
    "Changing important business/legal/factual information",
    "Acting only from the model score",
    "Treating the recommendation as causal proof"
]

print("Human review checks:", len(required_review_checks))
print("No-go actions:", len(no_go_actions))

print("\nHuman review is required before any content action.")
print("Automatic content changes are not allowed.")


Human review checks: 5
No-go actions: 5

Human review is required before any content action.
Automatic content changes are not allowed.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

The recommendation system should be monitored over time because search behavior, content quality, and the underlying data can change.

I would review the model when:

1. Model performance on newly observed labeled data drops meaningfully compared with the validated baseline.
2. The distribution of important input features changes substantially from the data used for validation.
3. The proportion of pages receiving each reason code changes unexpectedly.
4. The ranked queue begins producing recommendations that reviewers frequently reject.
5. The data pipeline or feature definitions change.

A retraining or re-validation decision should be based on measured evidence rather than a fixed assumption that the model will remain valid forever.

Before retraining, the data and target definition should be checked for leakage, data-quality problems, and changes in the decision context. Any new model should again be evaluated using an honest validation split.

In [4]:
monitoring_triggers = {
    "performance_drop": "Re-evaluate if performance drops on new labeled data",
    "feature_drift": "Investigate substantial changes in important inputs",
    "reason_code_shift": "Investigate unexpected changes in recommendation mix",
    "reviewer_rejection": "Investigate frequent human rejection of recommendations",
    "pipeline_change": "Re-validate after feature or data-pipeline changes"
}

print("Monitoring / retrain triggers:", len(monitoring_triggers))

for trigger, description in monitoring_triggers.items():
    print(f"- {trigger}: {description}")

print("\nRetraining should follow re-validation and a leakage/data-quality check.")

Monitoring / retrain triggers: 5
- performance_drop: Re-evaluate if performance drops on new labeled data
- feature_drift: Investigate substantial changes in important inputs
- reason_code_shift: Investigate unexpected changes in recommendation mix
- reviewer_rejection: Investigate frequent human rejection of recommendations
- pipeline_change: Re-validate after feature or data-pipeline changes

Retraining should follow re-validation and a leakage/data-quality check.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exported outputs

The ranked action queue will be exported so that the results can be reused in the final research paper.

The exported queue contains the content identifier, prioritization score, reason code, recommended review action, and supporting performance signals.

The output is intended for analysis and reporting. It should not be treated as an automated content-change instruction.

In [5]:
import os

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "content_action_queue.csv"
)

action_queue.to_csv(
    output_path,
    index=False
)

print("Export created:")
print(output_path)

print("\nRows exported:", len(action_queue))
print("Columns exported:", len(action_queue.columns))

print("\nTop 5 actions:")
display(action_queue.head(5))


Export created:
work/outputs/content_action_queue.csv

Rows exported: 30000
Columns exported: 8

Top 5 actions:


,content_id,action_score,reason_code,action,days_since_last_update,ctr,avg_position,rank
0,content_f6fdf87348f6,0.911907,STALE_AND_LOW_CTR,REVIEW_REFRESH,373,0.0,32.5,1
1,content_55a5b1c46474,0.911907,STALE_AND_LOW_CTR,REVIEW_REFRESH,373,0.0,7.5,2
2,content_1b4ec72dafd4,0.911857,STALE_AND_LOW_CTR,REVIEW_REFRESH,372,0.0,7.0,3
3,content_8d56efff1e71,0.911857,STALE_AND_LOW_CTR,REVIEW_REFRESH,372,0.0,35.0,4
4,content_06e19c6486b0,0.911797,STALE_AND_LOW_CTR,REVIEW_REFRESH,334,0.0,5.0,5


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.